# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) mass spectrometry dataset using the [mlcroissant](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is described by a Croissant schema at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Inspect available record sets, fields, and their `@id`. 

We use the Croissant specification's `record_set` property to list the available record sets with their `@id`, `name`, and the fields and columns within.

In [ ]:
# Parse and print all record sets and their fields by @id
def display_record_sets_info(ds):
    print("Available Record Sets:")
    for record_set in ds.metadata.record_sets:
        print(f"  RecordSet @id: {record_set.id}")
        print(f"    Name: {record_set.name}")
        if hasattr(record_set, 'fields'):
            print(f"    Fields:")
            for field in record_set.fields:
                print(f"      Field @id: {field.id}, name: {field.name}")
        if hasattr(record_set, 'columns'):
            print(f"    Columns:")
            for column in record_set.columns:
                print(f"      Column @id: {column.id}, name: {column.name}")
        print()

display_record_sets_info(dataset)

## 3. Data Extraction
Load data from each record set into a pandas DataFrame for analysis using their `@id`.

_You may inspect field and column names and ids in the previous step to select fields for further analysis._

In [ ]:
# Gather all record set @ids for extraction
record_set_ids = [record_set.id for record_set in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for RecordSet {record_set_id}")
    except Exception as e:
        print(f"Failed to load records for {record_set_id}: {e}")

# Show columns for the first record set
if record_set_ids:
    first_rs_id = record_set_ids[0]
    print(f"Columns for RecordSet {first_rs_id}:")
    print(dataframes[first_rs_id].columns.tolist())
    dataframes[first_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply basic data processing: filter on a numeric field (e.g., 'MW [kDa]' or similar), normalize, and group by some attribute, using field/column `@id` only.

In [ ]:
# Replace these with relevant @id values from the overview (adjust as needed to the dataset content)
record_set_id = record_set_ids[0]  # Use first record set for demonstration

# Attempt to auto-select a numeric column (by inferring from dtype)
df = dataframes[record_set_id]
numeric_field_id = None
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
        break

if numeric_field_id is not None:
    threshold = df[numeric_field_id].mean()  # As a sample threshold, mean
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered {len(filtered_df)} records where {numeric_field_id} > {threshold:.2f}")

    # Normalize the field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Attempt to select a grouping column that's likely categorical
    cat_field_id = None
    for col in df.columns:
        if df[col].dtype == object and col != numeric_field_id:
            cat_field_id = col
            break

    if cat_field_id:
        grouped = filtered_df.groupby(cat_field_id)[numeric_field_id].mean()
        print(f"Grouped means of {numeric_field_id} by {cat_field_id} (top 5):")
        print(grouped.head())
else:
    print("No numeric columns found for EDA.")

## 5. Visualization
Visualize the distribution of a selected numeric field and group means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if 'filtered_df' in locals() and cat_field_id:
        plt.figure(figsize=(10,5))
        sns.barplot(x=grouped.index[:10], y=grouped.values[:10])
        plt.title(f"Mean {numeric_field_id} by {cat_field_id} (top 10)")
        plt.xlabel(cat_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

- We used `mlcroissant` to load and inspect the FAIR² dataset mass spectrometry data package.
- Record sets, fields, and their Croissant `@id` values were used for precise referencing.
- A simple exploratory analysis demonstrates how to filter, normalize, group, and visualize records.

This notebook provides a starting point; for further analysis, consult the [Croissant schema specification](https://mlcommons.org/croissant/) and dataset [documentation](https://sen.science/doi/10.71728/senscience.vq4a-28xa).